# Multi-Dataset Horizon-Specific Channel Selection

This section implements the controlled predictive-utility experiment. Shared and horizon-adaptive source sets are compared under the matched evaluation protocol described in the paper.


In [1]:
from pathlib import Path
import gc
import time
import numpy as np
import pandas as pd
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"CUDA free: {free_b/1024**3:.2f} GiB / {total_b/1024**3:.2f} GiB")

PyTorch: 2.4.1+cu121
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
CUDA free: 70.25 GiB / 79.15 GiB


## 1. Configuration

In [2]:
DATA_ROOT = Path("/data/dataset")
DATASETS = {
    "PeMS03": DATA_ROOT / "PeMS03.npz",
    "PeMS04": DATA_ROOT / "PeMS04.npz",
    "PeMS07": DATA_ROOT / "PeMS07.npz",
    "PeMS08": DATA_ROOT / "PeMS08.npz",
}

OUTPUT_DIR = Path("./results_pems_multi_dataset_horizon_specific")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_IDX = 0
TRAIN_RATIO = 0.60
VAL_RATIO = 0.20
HISTORY_LEN = 12
PRED_LENS = [12, 24, 48]
TOPK_VALUES = [10, 20]
RIDGE_LAMBDAS = [1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0, 1000.0]

# PeMS07 has the largest channel count.
CHANNEL_BATCH = 8

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32

print("Device:", DEVICE)
print("History length:", HISTORY_LEN)
print("Channel batch:", CHANNEL_BATCH)

Device: cuda
History length: 12
Channel batch: 8


## 2. Verify and inspect dataset files

In [3]:
def load_dataset(path, feature_idx=0):
    npz = np.load(path)
    if "data" not in npz.files:
        raise KeyError(f"'data' key not found in {path}. Available keys: {npz.files}")
    raw = npz["data"]
    if raw.ndim == 3:
        x = raw[:, :, feature_idx]
    elif raw.ndim == 2:
        x = raw
    else:
        raise ValueError(f"Unexpected shape {raw.shape} in {path}")
    x = np.asarray(x, dtype=np.float32)
    if np.isnan(x).any():
        raise ValueError(f"NaNs detected in {path}")
    return raw.shape, x

for name, path in DATASETS.items():
    print(name, "->", path, "exists:", path.exists())
    if path.exists():
        raw_shape, xt = load_dataset(path, FEATURE_IDX)
        print(f"  raw={raw_shape}, selected={xt.shape}, min={xt.min():.3f}, max={xt.max():.3f}")

missing = [name for name, path in DATASETS.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing datasets: {missing}")

PeMS03 -> /data/dataset/PeMS03.npz exists: True
  raw=(26208, 358, 1), selected=(26208, 358), min=0.000, max=1852.000
PeMS04 -> /data/dataset/PeMS04.npz exists: True
  raw=(16992, 307, 3), selected=(16992, 307), min=0.000, max=919.000
PeMS07 -> /data/dataset/PeMS07.npz exists: True
  raw=(28224, 883, 1), selected=(28224, 883), min=0.000, max=1498.000
PeMS08 -> /data/dataset/PeMS08.npz exists: True
  raw=(17856, 170, 3), selected=(17856, 170), min=0.000, max=1147.000


## 3. Correlation and Top-K utilities

In [4]:
@torch.no_grad()
def horizon_cross_correlation(x_cpu, train_end, h, device):
    src = x_cpu[:train_end-h].to(device)
    tgt = x_cpu[h:train_end].to(device)

    src = src - src.mean(dim=0, keepdim=True)
    tgt = tgt - tgt.mean(dim=0, keepdim=True)

    src = src / src.square().sum(dim=0, keepdim=True).sqrt().clamp_min(1e-12)
    tgt = tgt / tgt.square().sum(dim=0, keepdim=True).sqrt().clamp_min(1e-12)

    # [target, source]
    corr = (src.T @ tgt).T.clamp(-1.0, 1.0).cpu()

    del src, tgt
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return corr

def exclude_self(score):
    score = score.clone()
    idx = torch.arange(score.shape[0])
    score[idx, idx] = -torch.inf
    return score

def build_shared_topk(correlations, H, K):
    score = torch.stack([correlations[h].abs() for h in range(1, H+1)], dim=0).mean(dim=0)
    score = exclude_self(score)
    return torch.topk(score, k=K, dim=1, largest=True, sorted=True).indices.cpu()

def build_specific_topk(correlations, H, K):
    out = {}
    for h in range(1, H+1):
        score = exclude_self(correlations[h].abs())
        out[h] = torch.topk(score, k=K, dim=1, largest=True, sorted=True).indices.cpu()
    return out

def make_origins(start, end, H, history_len):
    first_origin = start + history_len - 1
    last_origin_exclusive = end - H
    return torch.arange(first_origin, last_origin_exclusive, dtype=torch.long)

## 4. Temporal-history feature construction

In [5]:
@torch.no_grad()
def get_source_idx(selection, model, H, K, h, target_idx):
    if model == "self":
        return None
    if model == "shared":
        return selection[(H, K)]["shared"][target_idx]
    if model == "specific":
        return selection[(H, K)]["specific"][h][target_idx]
    raise ValueError(model)

@torch.no_grad()
def build_history_features(x_cpu, origins, target_idx, source_idx, history_len, device):
    lag_offsets = torch.arange(history_len - 1, -1, -1, dtype=torch.long)
    time_idx = origins.unsqueeze(1) - lag_offsets.unsqueeze(0)
    hist = x_cpu[time_idx]  # [N,L,C]

    target_hist = hist[:, :, target_idx].permute(2, 0, 1).contiguous()  # [B,N,L]
    parts = [target_hist]

    if source_idx is not None:
        src_hist = hist[:, :, source_idx]  # [N,L,B,K]
        src_hist = src_hist.permute(2, 0, 3, 1).contiguous()  # [B,N,K,L]
        B, N, K, L = src_hist.shape
        parts.append(src_hist.reshape(B, N, K * L))

    F = torch.cat(parts, dim=-1)
    del hist
    return F.to(device)

@torch.no_grad()
def build_target(x_cpu, origins, h, target_idx, device):
    y = x_cpu[origins + h][:, target_idx].T.contiguous()
    return y.to(device)

## 5. Robust batched Ridge regression

In [6]:
@torch.no_grad()
def compute_ridge_statistics(X, Y):
    B, N, F = X.shape
    ones = torch.ones((B, N, 1), dtype=X.dtype, device=X.device)
    Xb = torch.cat([X, ones], dim=-1)
    Xt = Xb.transpose(1, 2)
    return Xt @ Xb, Xt @ Y.unsqueeze(-1)

@torch.no_grad()
def solve_ridge(XtX, XtY, lam):
    B, D, _ = XtX.shape
    reg = torch.eye(D, dtype=XtX.dtype, device=XtX.device).unsqueeze(0).repeat(B, 1, 1)
    reg[:, -1, -1] = 0.0
    jitter = 1e-3 * torch.eye(D, dtype=XtX.dtype, device=XtX.device).unsqueeze(0)
    A = XtX + lam * reg + jitter
    try:
        return torch.linalg.solve(A, XtY).squeeze(-1)
    except RuntimeError as e:
        if "singular" not in str(e).lower():
            raise
        print(f"[WARN] Singular matrix at lambda={lam}; using pinv fallback.")
        return (torch.linalg.pinv(A) @ XtY).squeeze(-1)

@torch.no_grad()
def ridge_predict(X, W):
    B, N, F = X.shape
    ones = torch.ones((B, N, 1), dtype=X.dtype, device=X.device)
    Xb = torch.cat([X, ones], dim=-1)
    return (Xb @ W.unsqueeze(-1)).squeeze(-1)

## 6. Validation and test functions

In [7]:
@torch.no_grad()
def validate_configuration(x_cpu, C, origins, selection, model, H, K, history_len):
    train_orig = origins[H]["train"]
    val_orig = origins[H]["val"]
    stats = {lam: {"sse": 0.0, "sae": 0.0, "count": 0} for lam in RIDGE_LAMBDAS}

    for h in range(1, H+1):
        for c0 in range(0, C, CHANNEL_BATCH):
            c1 = min(c0 + CHANNEL_BATCH, C)
            target_idx = torch.arange(c0, c1, dtype=torch.long)
            source_idx = get_source_idx(selection, model, H, K, h, target_idx)

            Ftr = build_history_features(x_cpu, train_orig, target_idx, source_idx, history_len, DEVICE)
            ytr = build_target(x_cpu, train_orig, h, target_idx, DEVICE)
            Fva = build_history_features(x_cpu, val_orig, target_idx, source_idx, history_len, DEVICE)
            yva = build_target(x_cpu, val_orig, h, target_idx, DEVICE)

            XtX, XtY = compute_ridge_statistics(Ftr, ytr)
            for lam in RIDGE_LAMBDAS:
                W = solve_ridge(XtX, XtY, lam)
                pred = ridge_predict(Fva, W)
                err = pred - yva
                stats[lam]["sse"] += err.square().sum().item()
                stats[lam]["sae"] += err.abs().sum().item()
                stats[lam]["count"] += err.numel()
                del W, pred, err

            del Ftr, ytr, Fva, yva, XtX, XtY

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    rows = []
    for lam in RIDGE_LAMBDAS:
        d = stats[lam]
        rows.append({"lambda": lam, "val_mse": d["sse"]/d["count"], "val_mae": d["sae"]/d["count"]})
    df = pd.DataFrame(rows)
    best_lambda = float(df.loc[df.val_mse.idxmin(), "lambda"])
    return best_lambda, df

@torch.no_grad()
def test_configuration(x_cpu, C, origins, selection, model, H, K, history_len, best_lambda):
    train_orig = origins[H]["train"]
    val_orig = origins[H]["val"]
    test_orig = origins[H]["test"]
    fit_orig = torch.cat([train_orig, val_orig])
    step_rows = []

    for h in range(1, H+1):
        step_sse = step_sae = 0.0
        step_count = 0

        for c0 in range(0, C, CHANNEL_BATCH):
            c1 = min(c0 + CHANNEL_BATCH, C)
            target_idx = torch.arange(c0, c1, dtype=torch.long)
            source_idx = get_source_idx(selection, model, H, K, h, target_idx)

            Ffit = build_history_features(x_cpu, fit_orig, target_idx, source_idx, history_len, DEVICE)
            yfit = build_target(x_cpu, fit_orig, h, target_idx, DEVICE)
            Fte = build_history_features(x_cpu, test_orig, target_idx, source_idx, history_len, DEVICE)
            yte = build_target(x_cpu, test_orig, h, target_idx, DEVICE)

            XtX, XtY = compute_ridge_statistics(Ffit, yfit)
            W = solve_ridge(XtX, XtY, best_lambda)
            pred = ridge_predict(Fte, W)
            err = pred - yte

            step_sse += err.square().sum().item()
            step_sae += err.abs().sum().item()
            step_count += err.numel()
            del Ffit, yfit, Fte, yte, XtX, XtY, W, pred, err

        step_rows.append({"horizon_step": h, "test_mse": step_sse/step_count, "test_mae": step_sae/step_count})
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return pd.DataFrame(step_rows)

## 7. Run one dataset

In [8]:
def run_dataset(dataset_name, dataset_path):
    print("\n" + "#"*80)
    print(f"DATASET: {dataset_name}")
    print("#"*80)

    raw_shape, x = load_dataset(dataset_path, FEATURE_IDX)
    T, C = x.shape
    train_end = int(T * TRAIN_RATIO)
    val_end = int(T * (TRAIN_RATIO + VAL_RATIO))

    train_mean = x[:train_end].mean(axis=0, keepdims=True)
    train_std = np.maximum(x[:train_end].std(axis=0, keepdims=True), 1e-6)
    x_norm = (x - train_mean) / train_std
    x_cpu = torch.from_numpy(x_norm).to(dtype=DTYPE, device="cpu")

    print(f"T={T}, C={C}, raw_shape={raw_shape}")

    correlations = {}
    for h in range(1, max(PRED_LENS)+1):
        correlations[h] = horizon_cross_correlation(x_cpu, train_end, h, DEVICE)
    print("Correlations computed.")

    selection = {}
    for H in PRED_LENS:
        for K in TOPK_VALUES:
            selection[(H, K)] = {
                "shared": build_shared_topk(correlations, H, K),
                "specific": build_specific_topk(correlations, H, K),
            }

    origins = {}
    for H in PRED_LENS:
        origins[H] = {
            "train": make_origins(0, train_end, H, HISTORY_LEN),
            "val": make_origins(train_end, val_end, H, HISTORY_LEN),
            "test": make_origins(val_end, T, H, HISTORY_LEN),
        }

    results = []
    details = {}
    for H in PRED_LENS:
        configs = [("self", None), ("shared", 10), ("specific", 10), ("shared", 20), ("specific", 20)]
        for model, K in configs:
            print(f"\n{dataset_name}: H={H}, model={model}, K={K}")
            t0 = time.time()
            best_lambda, lambda_df = validate_configuration(
                x_cpu, C, origins, selection, model, H, K, HISTORY_LEN
            )
            test_df = test_configuration(
                x_cpu, C, origins, selection, model, H, K, HISTORY_LEN, best_lambda
            )
            summary = {
                "dataset": dataset_name, "history_len": HISTORY_LEN, "pred_len": H,
                "model": model, "K": K if K is not None else "-",
                "best_lambda": best_lambda,
                "test_mse": float(test_df.test_mse.mean()),
                "test_mae": float(test_df.test_mae.mean()),
            }
            results.append(summary)
            details[(dataset_name, H, model, K)] = {"lambda": lambda_df, "test": test_df}
            print(
                f"  TEST MSE={summary['test_mse']:.6f}, MAE={summary['test_mae']:.6f}, "
                f"lambda={best_lambda}, time={time.time()-t0:.1f}s"
            )
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    return results, details

## 8. Run all four datasets

In [9]:
all_results = []
all_details = {}

for dataset_name, dataset_path in DATASETS.items():
    dataset_results, dataset_details = run_dataset(dataset_name, dataset_path)
    all_results.extend(dataset_results)
    all_details.update(dataset_details)

    # Save after every dataset so partial progress is preserved.
    pd.DataFrame(all_results).to_csv(OUTPUT_DIR / "partial_results.csv", index=False)

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print("\nAll datasets finished.")


################################################################################
DATASET: PeMS03
################################################################################
T=26208, C=358, raw_shape=(26208, 358, 1)
Correlations computed.

PeMS03: H=12, model=self, K=None
  TEST MSE=0.109775, MAE=0.227379, lambda=10.0, time=58.4s

PeMS03: H=12, model=shared, K=10
  TEST MSE=0.087626, MAE=0.197515, lambda=100.0, time=106.9s

PeMS03: H=12, model=specific, K=10
  TEST MSE=0.084917, MAE=0.194001, lambda=100.0, time=109.5s

PeMS03: H=12, model=shared, K=20
  TEST MSE=0.085314, MAE=0.193311, lambda=100.0, time=157.0s

PeMS03: H=12, model=specific, K=20
  TEST MSE=0.083068, MAE=0.190639, lambda=100.0, time=157.8s

PeMS03: H=24, model=self, K=None
  TEST MSE=0.203496, MAE=0.318329, lambda=10.0, time=109.4s

PeMS03: H=24, model=shared, K=10
  TEST MSE=0.138704, MAE=0.248395, lambda=10.0, time=197.9s

PeMS03: H=24, model=specific, K=10
  TEST MSE=0.133942, MAE=0.245424, lambda=100.0, time=1

## 9. Main result table

In [10]:
results_df = pd.DataFrame(all_results)
display(results_df[[
    "dataset", "pred_len", "model", "K", "best_lambda", "test_mse", "test_mae"
]].round(6))
results_df.to_csv(OUTPUT_DIR / "multi_dataset_results.csv", index=False)

,dataset,pred_len,model,K,best_lambda,test_mse,test_mae
0,PeMS03,12,self,-,10.000,0.109775,0.227379
1,PeMS03,12,shared,10,100.000,0.087626,0.197515
2,PeMS03,12,specific,10,100.000,0.084917,0.194001
3,PeMS03,12,shared,20,100.000,0.085314,0.193311
4,PeMS03,12,specific,20,100.000,0.083068,0.190639
5,PeMS03,24,self,-,10.000,0.203496,0.318329
6,PeMS03,24,shared,10,10.000,0.138704,0.248395
7,PeMS03,24,specific,10,100.000,0.133942,0.245424
8,PeMS03,24,shared,20,100.000,0.128313,0.239215
9,PeMS03,24,specific,20,100.000,0.127965,0.238088


## 10. Shared vs Horizon-Specific comparison

In [11]:
comparison_rows = []
for dataset_name in DATASETS.keys():
    for H in PRED_LENS:
        for K in TOPK_VALUES:
            shared = results_df[
                (results_df.dataset == dataset_name) &
                (results_df.pred_len == H) &
                (results_df.model == "shared") &
                (results_df.K == K)
            ].iloc[0]
            specific = results_df[
                (results_df.dataset == dataset_name) &
                (results_df.pred_len == H) &
                (results_df.model == "specific") &
                (results_df.K == K)
            ].iloc[0]
            comparison_rows.append({
                "dataset": dataset_name, "pred_len": H, "K": K,
                "shared_mse": shared.test_mse, "specific_mse": specific.test_mse,
                "mse_improvement_%": (shared.test_mse-specific.test_mse)/shared.test_mse*100.0,
                "shared_mae": shared.test_mae, "specific_mae": specific.test_mae,
                "mae_improvement_%": (shared.test_mae-specific.test_mae)/shared.test_mae*100.0,
            })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(6))
comparison_df.to_csv(OUTPUT_DIR / "shared_vs_specific_comparison.csv", index=False)

,dataset,pred_len,K,shared_mse,specific_mse,mse_improvement_%,shared_mae,specific_mae,mae_improvement_%
0,PeMS03,12,10,0.087626,0.084917,3.091711,0.197515,0.194001,1.779408
1,PeMS03,12,20,0.085314,0.083068,2.632681,0.193311,0.190639,1.382196
2,PeMS03,24,10,0.138704,0.133942,3.433294,0.248395,0.245424,1.196117
3,PeMS03,24,20,0.128313,0.127965,0.271013,0.239215,0.238088,0.471142
4,PeMS03,48,10,0.239308,0.240856,-0.646774,0.339825,0.343266,-1.012791
5,PeMS03,48,20,0.228105,0.227262,0.369503,0.327573,0.330943,-1.028817
6,PeMS04,12,10,0.084857,0.084220,0.750261,0.197209,0.195854,0.687215
7,PeMS04,12,20,0.083017,0.082900,0.140978,0.194158,0.193363,0.409506
8,PeMS04,24,10,0.124791,0.125998,-0.967328,0.242814,0.243308,-0.203395
9,PeMS04,24,20,0.120465,0.120355,0.091381,0.237169,0.236093,0.453673


## 11. Dataset × Horizon MSE improvement summary

In [12]:
for K in TOPK_VALUES:
    print(f"\nK = {K}")
    pivot = comparison_df[comparison_df.K == K].pivot(
        index="dataset", columns="pred_len", values="mse_improvement_%"
    )
    display(pivot.round(3))
    pivot.to_csv(OUTPUT_DIR / f"mse_improvement_K{K}.csv")


K = 10


pred_len,12,24,48
dataset,,,
PeMS03,3.092,3.433,-0.647
PeMS04,0.750,-0.967,2.727
PeMS07,1.938,7.990,1.603
PeMS08,4.723,11.078,1.859



K = 20


pred_len,12,24,48
dataset,,,
PeMS03,2.633,0.271,0.370
PeMS04,0.141,0.091,2.732
PeMS07,1.571,41.537,2.034
PeMS08,0.647,31.986,-3.078


## 12. H=48 focus

In [13]:
h48_df = comparison_df[comparison_df.pred_len == 48].copy()
display(h48_df[[
    "dataset", "K", "shared_mse", "specific_mse", "mse_improvement_%", "mae_improvement_%"
]].round(6))

print("H=48 mean MSE improvement:", h48_df["mse_improvement_%"].mean())
print("H=48 positive cases:", int((h48_df["mse_improvement_%"] > 0).sum()), "/", len(h48_df))

,dataset,K,shared_mse,specific_mse,mse_improvement_%,mae_improvement_%
4,PeMS03,10,0.239308,0.240856,-0.646774,-1.012791
5,PeMS03,20,0.228105,0.227262,0.369503,-1.028817
10,PeMS04,10,0.234315,0.227926,2.726706,0.724226
11,PeMS04,20,0.211848,0.206060,2.732143,0.710650
16,PeMS07,10,0.263241,0.259022,1.602640,2.362307
17,PeMS07,20,0.240030,0.235148,2.033716,2.369434
22,PeMS08,10,0.247130,0.242535,1.859244,0.048929
23,PeMS08,20,0.219825,0.226591,-3.077771,-0.417396


H=48 mean MSE improvement: 0.9499257616193358
H=48 positive cases: 6 / 8


## 13. H=48 future-step-wise analysis

In [14]:
step_rows = []
H = 48
for dataset_name in DATASETS.keys():
    for K in TOPK_VALUES:
        shared_df = all_details[(dataset_name, H, "shared", K)]["test"]
        specific_df = all_details[(dataset_name, H, "specific", K)]["test"]
        for h in range(1, H+1):
            sm = shared_df.loc[shared_df.horizon_step == h, "test_mse"].iloc[0]
            qm = specific_df.loc[specific_df.horizon_step == h, "test_mse"].iloc[0]
            step_rows.append({
                "dataset": dataset_name, "K": K, "horizon_step": h,
                "shared_mse": sm, "specific_mse": qm,
                "mse_improvement_%": (sm-qm)/sm*100.0,
            })

step_df = pd.DataFrame(step_rows)
display(step_df[step_df.horizon_step.isin([1, 6, 12, 24, 48])].round(6))
step_df.to_csv(OUTPUT_DIR / "H48_future_step_improvement.csv", index=False)

,dataset,K,horizon_step,shared_mse,specific_mse,mse_improvement_%
0,PeMS03,10,1,0.041067,0.040944,0.298679
5,PeMS03,10,6,0.080426,0.085923,-6.835474
11,PeMS03,10,12,0.121305,0.122336,-0.849865
23,PeMS03,10,24,0.229526,0.226585,1.281483
47,PeMS03,10,48,0.429114,0.461083,-7.449849
48,PeMS03,20,1,0.040753,0.040464,0.708064
53,PeMS03,20,6,0.078968,0.086567,-9.622925
59,PeMS03,20,12,0.119734,0.119138,0.497630
71,PeMS03,20,24,0.224751,0.220206,2.022044
95,PeMS03,20,48,0.395054,0.413664,-4.710797


## 14. Lambda boundary check

In [15]:
lambda_boundary = results_df[results_df.best_lambda == 1000.0]
display(lambda_boundary)
print("Configurations hitting lambda=1000:", len(lambda_boundary))

,dataset,history_len,pred_len,model,K,best_lambda,test_mse,test_mae
43,PeMS07,12,48,shared,20,1000.0,0.240030,0.340617
44,PeMS07,12,48,specific,20,1000.0,0.235148,0.332546


Configurations hitting lambda=1000: 2


## 15. Interpretation

This section implements the controlled predictive-utility experiment. Shared and horizon-adaptive source sets are compared under the matched evaluation protocol described in the paper.
